In [ ]:
#!pip install tqdm
#!pip install statsmodels

In [30]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from statsmodels.stats.multitest import multipletests

In [31]:
dfm = pd.read_csv('culture/Culture_metadata_11-2-25_mods_taxids.txt', sep = '\t', dtype=str)
dfm['count'] = 1
dfm

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="species",
    values="count",
    fill_value=0
)

In [32]:
#filter df
GroupCol = "Biopsy_collection_date_year"
GroupA = "20"
GroupB = "26"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
results_df.to_csv(f"stats_out/fisher_culture_species_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|██████████████████| 181/181 [00:00<00:00, 196.67it/s]


In [33]:
dfm = pd.read_csv('culture/Culture_metadata_11-2-25_mods_taxids.txt', sep = '\t', dtype=str)
dfm['count'] = 1
dfm = dfm[dfm["Prog-Nonprog between 20-26y"] == 'N']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="species",
    values="count",
    fill_value=0
)

In [34]:
#filter df
GroupCol = "Biopsy_collection_date_year"
GroupA = "20"
GroupB = "26"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
results_df.to_csv(f"stats_out/fisher_Nonprog_culture_species_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|████████████████| 143/143 [00:00<00:00, 48148.47it/s]


In [35]:
dfm = pd.read_csv('culture/Culture_metadata_11-2-25_mods_taxids.txt', sep = '\t', dtype=str)
dfm['count'] = 1
dfm = dfm[dfm["Prog-Nonprog between 20-26y"] == 'P']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="species",
    values="count",
    fill_value=0
)

In [36]:
#filter df
GroupCol = "Biopsy_collection_date_year"
GroupA = "20"
GroupB = "26"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
results_df.to_csv(f"stats_out/fisher_Prog_culture_species_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|████████████████| 119/119 [00:00<00:00, 32688.60it/s]


In [37]:
dfm = pd.read_csv('culture/Culture_metadata_11-2-25_mods_taxids.txt', sep = '\t', dtype=str)
dfm['count'] = 1
dfm

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="genus",
    values="count",
    fill_value=0
)

In [38]:
#filter df
GroupCol = "Biopsy_collection_date_year"
GroupA = "20"
GroupB = "26"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
results_df.to_csv(f"stats_out/fisher_culture_genus_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|██████████████████| 60/60 [00:00<00:00, 41026.78it/s]


In [39]:
dfm = pd.read_csv('culture/Culture_metadata_11-2-25_mods_taxids.txt', sep = '\t', dtype=str)
dfm['count'] = 1
dfm = dfm[dfm["Prog-Nonprog between 20-26y"] == 'N']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="genus",
    values="count",
    fill_value=0
)

In [40]:
#filter df
GroupCol = "Biopsy_collection_date_year"
GroupA = "20"
GroupB = "26"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
results_df.to_csv(f"stats_out/fisher_Nonprog_culture_genus_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|██████████████████| 52/52 [00:00<00:00, 51695.62it/s]


In [41]:
dfm = pd.read_csv('culture/Culture_metadata_11-2-25_mods_taxids.txt', sep = '\t', dtype=str)
dfm['count'] = 1
dfm = dfm[dfm["Prog-Nonprog between 20-26y"] == 'P']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="genus",
    values="count",
    fill_value=0
)

In [42]:
#filter df
GroupCol = "Biopsy_collection_date_year"
GroupA = "20"
GroupB = "26"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
results_df.to_csv(f"stats_out/fisher_Prog_culture_genus_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|██████████████████| 43/43 [00:00<00:00, 32006.22it/s]


In [43]:
#########
#########
#########
#########

In [44]:
dfm = pd.read_csv('culture/Culture_metadata_11-2-25_mods_taxids.txt', sep = '\t', dtype=str)
dfm['count'] = 1
dfm

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="species",
    values="count",
    fill_value=0
)

In [45]:
#filter df
GroupCol = "Prog-Nonprog between 20-26y"
GroupA = "N"
GroupB = "P"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
results_df.to_csv(f"stats_out/fisher_culture_species_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|████████████████| 181/181 [00:00<00:00, 13660.02it/s]


In [46]:
dfm = pd.read_csv('culture/Culture_metadata_11-2-25_mods_taxids.txt', sep = '\t', dtype=str)
dfm['count'] = 1
dfm = dfm[dfm["Biopsy_collection_date_year"] == '20']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="species",
    values="count",
    fill_value=0
)

In [47]:
#filter df
GroupCol = "Prog-Nonprog between 20-26y"
GroupA = "N"
GroupB = "P"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
results_df.to_csv(f"stats_out/fisher_year_20_culture_species_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|████████████████| 146/146 [00:00<00:00, 37405.68it/s]


In [48]:
dfm = pd.read_csv('culture/Culture_metadata_11-2-25_mods_taxids.txt', sep = '\t', dtype=str)
dfm['count'] = 1
dfm = dfm[dfm["Biopsy_collection_date_year"] == '26']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="species",
    values="count",
    fill_value=0
)

In [49]:
#filter df
GroupCol = "Prog-Nonprog between 20-26y"
GroupA = "N"
GroupB = "P"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
results_df.to_csv(f"stats_out/fisher_year_26_culture_species_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|████████████████| 108/108 [00:00<00:00, 27807.54it/s]


In [50]:
dfm = pd.read_csv('culture/Culture_metadata_11-2-25_mods_taxids.txt', sep = '\t', dtype=str)
dfm['count'] = 1
dfm

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="genus",
    values="count",
    fill_value=0
)

In [51]:
#filter df
GroupCol = "Prog-Nonprog between 20-26y"
GroupA = "N"
GroupB = "P"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
results_df.to_csv(f"stats_out/fisher_culture_genus_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|██████████████████| 60/60 [00:00<00:00, 39119.89it/s]


In [52]:
dfm = pd.read_csv('culture/Culture_metadata_11-2-25_mods_taxids.txt', sep = '\t', dtype=str)
dfm['count'] = 1
dfm = dfm[dfm["Biopsy_collection_date_year"] == '20']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="genus",
    values="count",
    fill_value=0
)

In [53]:
#filter df
GroupCol = "Prog-Nonprog between 20-26y"
GroupA = "N"
GroupB = "P"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
results_df.to_csv(f"stats_out/fisher_year_20_culture_genus_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|██████████████████| 49/49 [00:00<00:00, 41603.42it/s]


In [54]:
dfm = pd.read_csv('culture/Culture_metadata_11-2-25_mods_taxids.txt', sep = '\t', dtype=str)
dfm['count'] = 1
dfm = dfm[dfm["Biopsy_collection_date_year"] == '26']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="genus",
    values="count",
    fill_value=0
)

In [55]:
#filter df
GroupCol = "Prog-Nonprog between 20-26y"
GroupA = "N"
GroupB = "P"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)


results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
results_df.to_csv(f"stats_out/fisher_year_26_culture_genus_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|██████████████████| 41/41 [00:00<00:00, 35603.82it/s]
